# Tutorial: IBM CLEAR with Eval Hub (local + deployed)

This notebook is a **walkthrough** of two paths: **Part A** runs the CLEAR adapter **locally** (same as **`docs/03-local-run.md`**). **Part B** shows a **deployed** Eval Hub (**`curl`** and optional Python SDK), aligned with **`docs/04-deployed-eval-hub.md`**. For the full narrative see **[`README.md`](README.md)** and **`docs/`** in this folder.

Example outputs from a small run (open the HTML in a browser): **[`output/local/clear_results.html`](output/local/clear_results.html)**, **[`output/local/clear_results.json`](output/local/clear_results.json)**. Example traces: **`input-traces/`** (see **`docs/02-agent-traces.md`**).


## Prerequisites (Part A: local run)

1. **Adapter virtualenv.** From **`adapters/clear`** (next to **`main.py`**), create **`.venv`** and run **`pip install -r requirements.txt`** (CLEAR installs from Git per that file; see **`docs/03-local-run.md`**).
2. **`.env`.** Copy **`examples/env.example`** to **`examples/.env`**. Set **`MODEL_URL`** (OpenAI compatible API base, often **`…/v1`**) and **`MODEL_NAME`**. Set **`OPENAI_API_KEY`** only if your endpoint requires it.
3. **MLflow (optional).** To upload artifacts to MLflow, set **`MLFLOW_TRACKING_URI`** and experiment naming as in **`docs/03-local-run.md`** and the adapter README.
4. **Traces.** CLEAR-compatible **`*.json`** under **`examples/input-traces/`** (see **`docs/02-agent-traces.md`** for the current sample file and MLflow shape).

Install **`python-dotenv`** in this Jupyter kernel so **Step 3** can load **`.env`** (`pip install python-dotenv`).


## Step 1: Resolve paths (run first)

Run from **`adapters/clear`** or **`adapters/clear/examples`**. This cell sets **`ADAPTER_ROOT`**, **`INPUT_TRACES`** (`examples/input-traces/`), **`OUTPUT_DIR_LOCAL`** (`examples/output/local/`), and **`OUTPUT_DIR_EVALHUB`** (used in Part B).

In [1]:
from pathlib import Path
import json
import os
import sys
import subprocess
import time

HERE = Path.cwd().resolve()
if HERE.name == "examples" and (HERE.parent / "main.py").is_file():
    ADAPTER_ROOT = HERE.parent
    EXAMPLES_DIR = HERE
elif (HERE / "main.py").is_file():
    ADAPTER_ROOT = HERE
    EXAMPLES_DIR = HERE / "examples"
else:
    raise RuntimeError(
        "Use working directory adapters/clear or adapters/clear/examples (folder containing main.py)."
    )

INPUT_TRACES = EXAMPLES_DIR / "input-traces"
_out_root = EXAMPLES_DIR / "output"
OUTPUT_DIR_LOCAL = _out_root / "local"   # Part A: main.py on this machine
OUTPUT_DIR_EVALHUB = _out_root / "evalhub"  # Part B: SDK snapshots from deployed Eval Hub
OUTPUT_DIR_LOCAL.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR_EVALHUB.mkdir(parents=True, exist_ok=True)

# Part A runs main.py in a subprocess; use adapter .venv if it exists (CLEAR lives there).
_venv_py = ADAPTER_ROOT / ".venv" / "bin" / "python"
MAIN_PYTHON = str(_venv_py) if _venv_py.is_file() else sys.executable

print("Adapter root:", ADAPTER_ROOT)
print("Examples dir:", EXAMPLES_DIR)
print("Input traces:", INPUT_TRACES)
print("Part A output (local CLEAR):", OUTPUT_DIR_LOCAL)
print("Part B output (Eval Hub SDK files):", OUTPUT_DIR_EVALHUB)
print("Python for main.py subprocess:", MAIN_PYTHON)
if not _venv_py.is_file():
    print(
        "No .venv at adapter root; main.py uses the Jupyter kernel Python.\n"
        "Fix: cd adapters/clear && python3 -m venv .venv && .venv/bin/pip install -r requirements.txt\n"
        "Or: register that venv as your Jupyter kernel and select it here."
    )

Adapter root: /Users/supathak/clear-evalhub-integration/eval-hub-contrib/adapters/clear
Examples dir: /Users/supathak/clear-evalhub-integration/eval-hub-contrib/adapters/clear/examples
Input traces: /Users/supathak/clear-evalhub-integration/eval-hub-contrib/adapters/clear/examples/input-traces
Part A output (local CLEAR): /Users/supathak/clear-evalhub-integration/eval-hub-contrib/adapters/clear/examples/output/local
Part B output (Eval Hub SDK files): /Users/supathak/clear-evalhub-integration/eval-hub-contrib/adapters/clear/examples/output/evalhub
Python for main.py subprocess: /Users/supathak/clear-evalhub-integration/eval-hub-contrib/adapters/clear/.venv/bin/python


### Virtualenv and subprocess

**Part A** runs **`main.py`** with **`adapters/clear/.venv/bin/python`** when that venv exists (same as the CLI). Create the venv once from **`adapters/clear`**: **`python3 -m venv .venv`** then **`pip install -r requirements.txt`**. Re-run **Step 1** after creating it so **Python for main.py subprocess** prints the venv path.

---

## Part A: Local run

Same flow as **`docs/03-local-run.md`**. We load **`meta/job.json`** (default **`benchmark_id`**: **`agentic-evaluation`**), point **`data_dir`** at **`examples/input-traces/`**, set **`MODEL_URL`** / **`MODEL_NAME`** from **`.env`**, drop **`test_data_ref`** (cluster staging only), and run **`main.py`** with **`EVALHUB_MODE=local`** and **`EVALHUB_JOB_SPEC_PATH`** set to the written **`local-job-spec.json`**.

In a terminal you would run:

```bash
cd adapters/clear
export EVALHUB_MODE=local
export EVALHUB_JOB_SPEC_PATH=/absolute/path/to/local-job-spec.json
# Optional: export MLFLOW_TRACKING_URI=...
python main.py
```

The **Step 3** cell does the same **`export`** values when it launches **`main.py`**. **`Connection refused`** on **`callback_url`** in the log is normal locally.

In [2]:
# Step 2: traces must exist under input-traces/ (top-level *.json only; use rglob if nested)
traces = sorted(INPUT_TRACES.glob("*.json"))
if not traces:
    raise FileNotFoundError(
        f"Add at least one *.json trace file under {INPUT_TRACES}."
    )
print(f"✓ Found {len(traces)} trace file(s):", [p.name for p in traces])

✓ Found 2 trace file(s): ['tr-236907d3019ea2a5f8e81baeb258753b.json', 'tr-e08208946684c866332b7416670bad41.json']


### `.env` variables used in Step 3

**Step 3** calls **`load_dotenv(examples/.env)`** when **`python-dotenv`** is installed. **`MODEL_URL`** / **`MODEL_NAME`** override **`meta/job.json`**. **`EXPERIMENT_NAME`** sets **`job["experiment_name"]`** so the adapter can upload to MLflow when **`MLFLOW_TRACKING_URI`** is also set (same as **`docs/03-local-run.md`**).


In [3]:
# Step 3: job spec from meta/job.json, then run main.py
import time

try:
    from dotenv import load_dotenv

    load_dotenv(EXAMPLES_DIR / ".env")
except ImportError:
    pass

meta_job = ADAPTER_ROOT / "meta" / "job.json"
job = json.loads(meta_job.read_text(encoding="utf-8"))

# --- Tune locally (same shape as meta/job.json) ---
USE_MLFLOW = False  # If False and no EXPERIMENT_NAME in .env, experiment fields are cleared (no MLflow upload).
INFERENCE_BACKEND = "litellm"  # see docs/03-local-run.md

job["id"] = "clear-notebook-local-001"
job["parameters"] = dict(job["parameters"])
job["parameters"]["data_dir"] = str(INPUT_TRACES.resolve())
job["parameters"]["results_dir"] = str((EXAMPLES_DIR / "output").resolve())
job["parameters"]["run_name"] = "local"
job["parameters"]["inference_backend"] = INFERENCE_BACKEND
job.pop("test_data_ref", None)

_exp_name = (os.getenv("EXPERIMENT_NAME") or "").strip()
if _exp_name:
    job["experiment_name"] = _exp_name
    print("Job experiment_name (MLflow):", _exp_name)
elif not USE_MLFLOW:
    job["experiment_name"] = None
    job["parameters"].pop("mlflow_experiment_name", None)

# Local run: Kubernetes secret refs in meta/job.json are not mounted; drop for laptop/Ollama.
if job.get("model", {}).get("auth"):
    job["model"] = dict(job["model"])
    job["model"].pop("auth", None)

_murl = os.getenv("MODEL_URL")
_mname = os.getenv("MODEL_NAME")
if _murl or _mname:
    job["model"] = dict(job.get("model") or {})
    if _murl:
        job["model"]["url"] = _murl.strip()
    if _mname:
        _mn = _mname.strip()
        job["model"]["name"] = _mn
        job["parameters"]["eval_model_name"] = _mn
    print("Applied from .env:", {k: v for k, v in [("MODEL_URL", _murl), ("MODEL_NAME", _mname)] if v})

job_path = OUTPUT_DIR_LOCAL / "local-job-spec.json"
job_path.write_text(json.dumps(job, indent=2), encoding="utf-8")

env = os.environ.copy()
env["EVALHUB_MODE"] = "local"
env["EVALHUB_JOB_SPEC_PATH"] = str(job_path.resolve())

log_path = OUTPUT_DIR_LOCAL / "local-main-py.log"
_t0 = time.perf_counter()
with log_path.open("w", encoding="utf-8") as logf:
    proc = subprocess.run(
        [MAIN_PYTHON, str(ADAPTER_ROOT / "main.py")],
        cwd=str(ADAPTER_ROOT),
        env=env,
        stdout=logf,
        stderr=subprocess.STDOUT,
        text=True,
    )

_elapsed_s = time.perf_counter() - _t0
print(f"✓ Wrote job spec: {job_path}")
print(f"✓ Wrote log: {log_path}")
print(f"✓ Exit code: {proc.returncode} (see log if non-zero)")
print(f"✓ Local run wall time: {_elapsed_s:.2f}s (main.py subprocess)")
if proc.returncode != 0:
    print("WARNING: main.py failed. Open local-main-py.log for details.")

Job experiment_name (MLflow): ibm-clear-agentic-nerc-beta-test
Applied from .env: {'MODEL_URL': 'http://127.0.0.1:11434/v1', 'MODEL_NAME': 'gpt-oss:20b'}
✓ Wrote job spec: /Users/supathak/clear-evalhub-integration/eval-hub-contrib/adapters/clear/examples/output/local/local-job-spec.json
✓ Wrote log: /Users/supathak/clear-evalhub-integration/eval-hub-contrib/adapters/clear/examples/output/local/local-main-py.log
✓ Exit code: 0 (see log if non-zero)
✓ Local run wall time: 166.94s (main.py subprocess)


In [4]:
# Step 4: confirm CLEAR outputs (JSON and HTML)
final_json = OUTPUT_DIR_LOCAL / "clear_results.json"
html_here = sorted(OUTPUT_DIR_LOCAL.glob("*.html"))
out_root = EXAMPLES_DIR / "output"
html_under_output = sorted(out_root.rglob("clear_results.html"))

if final_json.is_file():
    print("CLEAR wrote JSON:", final_json)
else:
    print("No clear_results.json in output/local yet. Read local-main-py.log for errors.")
if html_here:
    print("HTML in output/local:", [p.name for p in html_here])
elif html_under_output:
    print("Found dashboard HTML under output/:", [str(p.relative_to(out_root)) for p in html_under_output[:5]])
else:
    print("No dashboard HTML found yet; check local-main-py.log and tree under examples/output/.")


CLEAR wrote JSON: /Users/supathak/clear-evalhub-integration/eval-hub-contrib/adapters/clear/examples/output/local/clear_results.json
HTML in output/local: ['clear_results.html']


## Prerequisites (Part B)

You need a **reachable Eval Hub HTTPS route** (`EVALHUB_URL`), **Bearer token** (`EVALHUB_TOKEN`), **tenant** (`EVALHUB_TENANT`), and **`ibm-clear`** registered on that Hub. Set **`NAMESPACE`** if you use `oc get route evalhub -n ...` to discover the URL.

Fill **`examples/.env`** from **`env.example`**: model (`MODEL_*`), S3 trace location (`S3_*`), optional **`EXPERIMENT_NAME`**. Do not paste secrets into cells.

```bash
export EVALHUB_TOKEN=$(oc whoami -t)
export EVALHUB_URL=https://$(oc get route evalhub -n "${NAMESPACE}" -o jsonpath='{.spec.host}')
export EVALHUB_TENANT="your-tenant"
```

Tenants: https://eval-hub.github.io/development/multi-tenancy

**SDK:** run the **`%pip`** cell below, **restart the kernel**, then Step 1 again before importing **`evalhub`**.


---

# Part B: Deployed Eval Hub

**What we do:** submit a job like **`meta/job.json`**: traces come from **S3** (`test_data_ref`), the worker runs **`ibm-clear`**, results show up in the Hub (and optionally MLflow if the job sets an experiment).

1. **Upload** CLEAR-compatible **`*.json`** under your bucket prefix (e.g. `aws s3 sync ./traces/ s3://$S3_BUCKET/$S3_PREFIX`).
2. **Shell:** `source` or `export` variables from **`examples/.env`**, then run **`curl`** (or use the SDK cells below; they write under **`examples/output/evalhub/`**).

## `curl` example (aligned with `meta/job.json`)

`test_data_ref.s3` uses **`path`** and **`secret_ref`** as **`name` + `namespace`** (same shape as **`adapters/clear/meta/job.json`**). Adjust field names if your Eval Hub API differs.

Optional **`parameters.clear_dashboard_theme`**: e.g. **`red_hat`** (branded HTML) or **`clear`** (stock CLEAR). See **`docs/06-dashboard-theme.md`**. Omit to use the adapter default.

```bash
export TOKEN="${EVALHUB_TOKEN:-$(oc whoami -t)}"
# EVALHUB_URL, EVALHUB_TENANT, MODEL_*, S3_*, EXPERIMENT_NAME from examples/.env

curl -s -k -X POST \
  "${EVALHUB_URL}/api/v1/evaluations/jobs" \
  -H "Authorization: Bearer ${TOKEN}" \
  -H "X-Tenant: ${EVALHUB_TENANT}" \
  -H "Content-Type: application/json" \
  -d @- <<EOF
{
  "name": "clear-agentic-example",
  "model": {
    "url": "${MODEL_URL}",
    "name": "${MODEL_NAME}"
  },
  "experiment": {
    "name": "${EXPERIMENT_NAME:-clear-agentic-mlflow-exp}"
  },
  "benchmarks": [
    {
      "id": "agentic-evaluation",
      "provider_id": "ibm-clear",
      "parameters": {
        "eval_model_name": "${MODEL_NAME}",
        "provider": "openai",
        "agent_framework": "langgraph",
        "observability_framework": "mlflow",
        "inference_backend": "litellm",
        "agent_mode": true,
        "clear_dashboard_theme": "${CLEAR_DASHBOARD_THEME:-red_hat}"
      },
      "test_data_ref": {
        "s3": {
          "bucket": "${S3_BUCKET}",
          "path": "${S3_PREFIX}",
          "secret_ref": {
            "name": "${S3_SECRET_NAME}",
            "namespace": "${S3_SECRET_NAMESPACE}"
          }
        }
      }
    }
  ]
}
EOF
```

Worker needs **`OPENAI_API_KEY`** (or platform secret) if **`model.url`** requires it.

## Optional: Python SDK

Same job shape as above: connect, list providers, submit, wait; outputs under **`examples/output/evalhub/`**.


**Install SDK into this kernel:** run the cell below, then **Kernel → Restart Kernel**, then **Step 1** → load **`.env`** → **Import SDK** cells.

If **`evalhub`** is missing, pick the same **`.venv`** as in the terminal (**Python: Select Interpreter**), restart, re-run this cell.


In [5]:
%pip install -q -U typing_extensions python-dotenv
import subprocess
import sys

try:
    EXAMPLES_DIR
except NameError:
    raise RuntimeError("Run Step 1 first so EXAMPLES_DIR is defined.")

_repo = EXAMPLES_DIR.parent.parent.parent.parent
_sdk = _repo / "eval-hub-sdk"
if _sdk.is_dir():
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "-e", f"{_sdk}[adapter]"]
    )
    print("Installed eval-hub-sdk (editable from", _sdk, ")")
else:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "eval-hub-sdk[adapter]==0.1.7",
        ]
    )
    print("Installed eval-hub-sdk from PyPI")

print()
print(">>> Now: Kernel → Restart Kernel, then re-run Step 1, load .env, then Import SDK.")


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Installed eval-hub-sdk (editable from /Users/supathak/clear-evalhub-integration/eval-hub-sdk )

>>> Now: Kernel → Restart Kernel, then re-run Step 1, load .env, then Import SDK.



[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip


Load **`.env`** after **Step 1** (needs **`EXAMPLES_DIR`**). Copy from **`env.example`** if you have not already.

In [ ]:
try:
    from dotenv import load_dotenv
except ImportError:
    raise ImportError('Install python-dotenv: pip install python-dotenv')

_env_ok = load_dotenv(EXAMPLES_DIR / ".env")
print("Loaded examples/.env" if _env_ok else "No .env: copy examples/env.example to examples/.env and fill values (see Part B prerequisites)")

Loaded examples/.env


## Part B: Import SDK and read credentials

Only run this **after** the install cell above **and** a **kernel restart**.

In [7]:
try:
    from evalhub import (
        SyncEvalHubClient,
        JobSubmissionRequest,
        BenchmarkConfig,
        ModelConfig,
        JobStatus,
    )
except ModuleNotFoundError as _e:
    raise ModuleNotFoundError(
        "Package 'evalhub' is not installed in THIS Jupyter kernel. "
        "1) Run the install cell above (with Step 1 done first). "
        "2) Kernel → Restart Kernel. "
        "3) Re-run Step 1, load .env, then this cell. "
        "4) Ensure the notebook kernel matches your .venv (Python: Select Interpreter). "
        "Or in a terminal: python -m pip install 'eval-hub-sdk[adapter]==0.1.7' using the same Python as the kernel."
    ) from _e

evalhub_token = os.getenv("EVALHUB_TOKEN")
if not evalhub_token:
    raise ValueError(
        "EVALHUB_TOKEN is not set. Copy examples/env.example to examples/.env and set EVALHUB_* (see Prerequisites)."
    )

evalhub_url = os.getenv("EVALHUB_URL")
if not evalhub_url:
    raise ValueError("EVALHUB_URL is not set. Set it in examples/.env (from env.example); see Prerequisites.")

evalhub_tenant = os.getenv("EVALHUB_TENANT")
if not evalhub_tenant:
    raise ValueError("EVALHUB_TENANT is not set. Set it in examples/.env (from env.example); see Prerequisites.")

ValueError: EVALHUB_TOKEN is not set. Copy examples/env.example to examples/.env and set EVALHUB_* (see Prerequisites).

## Connect to Eval Hub (health check)

In [ ]:
client = SyncEvalHubClient(
    base_url=evalhub_url,
    auth_token=evalhub_token,
    insecure=True,
    tenant=evalhub_tenant,
)
try:
    health = client.health()
    status = getattr(health, "status", None)
    if status is None and isinstance(health, dict):
        status = health.get("status")
    print(f"✓ Eval Hub health: {status}")
    extra = health.model_dump() if hasattr(health, "model_dump") else health
    if isinstance(extra, dict):
        print(f"  Version: {extra.get('version', 'unknown')}")
        print(f"  Uptime: {extra.get('uptime_seconds', 0):.1f}s")
except Exception as e:
    print(f"✗ Failed to connect: {e}")

## List providers and benchmarks

In [ ]:
providers = client.providers.list()
print(f"✓ Found {len(providers)} providers:")
for provider in providers:
    print(f"  - {provider.resource.id}: {provider.name}")

### We use `ibm-clear` as `provider_id` in this notebook

List benchmarks for that provider (CLEAR exposes **`agentic-evaluation`**).

In [ ]:
provider_id = "ibm-clear"

In [ ]:
try:
    benchmarks = client.benchmarks.list(provider_id=provider_id)
    print(f"\n✓ Found {len(benchmarks)} benchmarks for {provider_id}:")
    for benchmark in benchmarks:
        print(f"  - {benchmark.id}: {benchmark.name}")
except Exception as e:
    print(f"Could not list benchmarks (provider may not be registered yet): {e}")

## Submit a job (SDK)

Uses **`MODEL_*`**, **`S3_*`**, **`EXPERIMENT_NAME`**, **`JOB_NAME`** from **`.env`**. Optional **`CLEAR_DASHBOARD_THEME`** (`red_hat`, `clear`, …) matches **`parameters.clear_dashboard_theme`** in **`meta/job.json`**. In-cluster model URL is often `http://<svc>.<ns>.svc.cluster.local:.../v1` (`oc get svc`).


In [ ]:
# MODEL_URL / MODEL_NAME from examples/.env (same as Part A); defaults are placeholders only.
model_url = os.getenv(
    "MODEL_URL",
    "http://YOUR_SVC.YOUR_NAMESPACE.svc.cluster.local:8000/v1",
)
model_name = os.getenv("MODEL_NAME", "example-model")
job_name = os.getenv("JOB_NAME", "clear-evalhub-job-1")
benchmark_id = "agentic-evaluation"

In [ ]:
clear_params = {
    "eval_model_name": model_name,
    "provider": "openai",
    "agent_framework": "langgraph",
    "observability_framework": "mlflow",
    "inference_backend": "litellm",
    "agent_mode": True,
}
_theme = (os.getenv("CLEAR_DASHBOARD_THEME") or "").strip()
if _theme:
    clear_params["clear_dashboard_theme"] = _theme

s3_bucket = os.getenv("S3_BUCKET", "").strip()
s3_prefix = os.getenv("S3_PREFIX", "").strip()
sn = os.getenv("S3_SECRET_NAME", "").strip()
sns = os.getenv("S3_SECRET_NAMESPACE", "").strip()
legacy_ref = os.getenv("S3_SECRET_REF", "").strip()

if s3_bucket and s3_prefix and sn and sns:
    test_data_ref = {
        "s3": {
            "bucket": s3_bucket,
            "path": s3_prefix,
            "secret_ref": {"name": sn, "namespace": sns},
        }
    }
elif s3_bucket and s3_prefix and legacy_ref:
    test_data_ref = {
        "s3": {
            "bucket": s3_bucket,
            "key": s3_prefix,
            "secret_ref": legacy_ref,
        }
    }
else:
    raise ValueError(
        "Set S3_BUCKET and S3_PREFIX. For meta/job.json shape use S3_SECRET_NAME + "
        "S3_SECRET_NAMESPACE, or set S3_SECRET_REF for key + string secret_ref."
    )
_exp = os.getenv("EXPERIMENT_NAME", "clear-agentic-mlflow-exp").strip()
experiment = {"name": _exp} if _exp else None

job_request = JobSubmissionRequest(
    name=job_name,
    experiment=experiment,
    model=ModelConfig(url=model_url, name=model_name),
    benchmarks=[
        BenchmarkConfig(
            id=benchmark_id,
            provider_id=provider_id,
            parameters=clear_params,
            test_data_ref=test_data_ref,
        )
    ],
)

job = client.jobs.submit(job_request)
submitted_job_id = job.id
print("✓ Job submitted successfully")
print(f"  Job ID: {submitted_job_id}")
print(f"  State: {job.state}")
print(f"  Created: {job.resource.created_at}")

(OUTPUT_DIR_EVALHUB / "evalhub-submit-response.json").write_text(
    job.model_dump_json(indent=2), encoding="utf-8"
)
print(f"  Saved: {OUTPUT_DIR_EVALHUB / 'evalhub-submit-response.json'}")


In [ ]:
updated_job = client.jobs.get(submitted_job_id)
print(f"\n✓ Current job state: {updated_job.state}")
if updated_job.status and updated_job.status.message:
    print(f"  Message: {updated_job.status.message.message}")

## Wait for the job to complete

In [ ]:
while updated_job.state not in (JobStatus.COMPLETED, JobStatus.FAILED):
    print(f"\n✓ Current job state: {updated_job.state}")
    time.sleep(5)
    updated_job = client.jobs.get(submitted_job_id)

if updated_job.state == JobStatus.FAILED:
    if updated_job.status and updated_job.status.message:
        print(f"Job failed: {updated_job.status.message.message}")
    else:
        print("Job failed: Unknown")
else:
    print("✓ Job completed successfully")

## Job results (CLEAR metrics)

Eval Hub exposes metrics under `updated_job.results.benchmarks[0].metrics` (see `provider.yaml`).


In [ ]:
if updated_job.results and updated_job.results.benchmarks:
    m = updated_job.results.benchmarks[0].metrics
    print("CLEAR / benchmark metrics:", m)
    (OUTPUT_DIR_EVALHUB / "evalhub-clear-metrics.json").write_text(
        json.dumps(m, indent=2, default=str), encoding="utf-8"
    )
    print(f"✓ Wrote {OUTPUT_DIR_EVALHUB / 'evalhub-clear-metrics.json'}")
else:
    print("No benchmark results on job object yet.")

In [ ]:
full_path = OUTPUT_DIR_EVALHUB / "evalhub-job-full.json"
full_path.write_text(updated_job.model_dump_json(indent=2), encoding="utf-8")
print("Wrote", full_path)

## Clean up (optional)

**`client.jobs.cancel(..., hard_delete=True)`** permanently **removes the evaluation job** from Eval Hub (not just “stop running”). Leave the line **commented** unless you really want to delete that job record.

**`client.close()`** closes the SDK’s HTTP session to Eval Hub, good hygiene when you are done with Part B.

In [ ]:
# Uncomment to permanently delete this job on Eval Hub (hard_delete=True)
# client.jobs.cancel(submitted_job_id, hard_delete=True)

## Close the client

In [ ]:
try:
    client.close()
except NameError:
    pass

## What gets saved where

| Where | Artifacts |
|-------|-----------|
| **Part A** (`examples/output/local/`) | **`clear_results.json`**, HTML, **`local-job-spec.json`**, log |
| **Part B SDK** (`examples/output/evalhub/`) | Submit response, metrics JSON, full job JSON (not the pod’s full **`clear_results.json`** unless you add MLflow/OCI on the job) |
| **Cluster workload** | Full CLEAR outputs inside the pod; retain via MLflow/OCI if needed. See adapter **`README.md`**. |
